# 03 · AI as Your Analyst, and How to Fact-Check It
**Presentation & Data Storytelling with AI Tools** · Day 1 · Module 3 · about 30 minutes

AI assistants can find patterns in a dataset in seconds. They can also state a wrong number with total confidence. In this notebook you will:
1. Load a realistic (fictional) hospital emergency-room dataset.
2. Compute an **answer key** with code: numbers you can trust.
3. Send a well-built prompt to an AI (Gemini right here in Colab, or Claude / ChatGPT).
4. **Fact-check** what the AI told you against the answer key.
5. Turn one verified insight into a story chart.

▶ Run each cell with **Shift + Enter**. Only change lines marked ✏️.

> ⚠️ This dataset is **fictional**. Never paste real patient, customer or employee data into a public AI tool unless your organization has approved that tool for that data.

In [ ]:
# ▶ Tools, story style, and the data generator
import pandas as pd
import numpy as np

# ▶ STORY STYLE — run once. Every chart after this cell gets the clean look.
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

NAVY, CORAL, GRAY, LIGHT, INK = "#1B2A41", "#FF6F59", "#BFC5CF", "#EEF1F5", "#4A5568"
plt.rcParams.update({
    "figure.figsize": (10, 5.6), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": LIGHT, "grid.linewidth": 1,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "axes.axisbelow": True,
    "xtick.color": INK, "ytick.color": INK, "ytick.left": False,
    "font.size": 11, "axes.titlesize": 16, "axes.titleweight": "bold",
})

def add_titles(ax, title, subtitle=""):
    """Headline (the insight) + a smaller subtitle (what is measured), left-aligned."""
    ax.set_title(title, loc="left", pad=34 if subtitle else 12, color=NAVY)
    if subtitle:
        ax.text(0, 1.035, subtitle, transform=ax.transAxes, fontsize=11, color=INK, va="bottom")

def add_source(fig, text):
    """Small source line in the bottom-left corner: always say where data came from."""
    fig.text(0.01, -0.02, text, fontsize=9, color="#8A94A6", ha="left", va="top")

print("Story style loaded ✔")

def make_er_data(seed=2025):
    """Fictional 'Riverside Community Hospital' emergency-room visits for 2025 (one row per visit)."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
    dow_mult = np.array([1.17, 1.04, 0.99, 0.97, 1.00, 0.90, 0.93])            # Mon ... Sun
    month_mult = np.array([1.14, 1.10, 1.00, 0.97, 0.98, 1.00, 1.03, 1.01, 0.98, 1.00, 1.04, 1.12])
    daily_n = rng.poisson(92 * dow_mult[dates.dayofweek] * month_mult[dates.month - 1])
    date = np.repeat(dates.values, daily_n)
    n = len(date)

    hour_w = np.array([2.0, 1.6, 1.3, 1.1, 1.0, 1.1, 1.6, 2.4, 3.4, 4.4, 5.0, 5.3,
                       5.3, 5.2, 5.1, 5.1, 5.2, 5.4, 5.6, 5.7, 5.3, 4.6, 3.7, 2.8])
    hour = rng.choice(24, size=n, p=hour_w / hour_w.sum())
    triage = rng.choice([1, 2, 3, 4, 5], size=n, p=[0.02, 0.13, 0.46, 0.31, 0.08])
    amb_p = np.array([0, 0.75, 0.45, 0.20, 0.06, 0.03])[triage]
    arrival = np.where(rng.random(n) < amb_p, "Ambulance", "Walk-in")
    age = rng.choice(["0-17", "18-44", "45-64", "65+"], size=n, p=[0.18, 0.37, 0.25, 0.20])

    crowd_day = np.repeat(daily_n / daily_n.mean(), daily_n)
    crowd_hour = hour_w[hour] / hour_w.mean()
    base_wait = np.array([0, 2, 12, 38, 52, 48])[triage]
    wait = base_wait * (0.55 + 0.45 * crowd_day) * (0.6 + 0.4 * crowd_hour) * rng.gamma(4, 0.25, n)
    wait = np.maximum(0, wait.round()).astype(int)

    lwbs_p = np.where(triage >= 3, np.clip(0.005 + 0.0012 * (wait - 30), 0.005, 0.40), 0.0)
    lwbs = rng.random(n) < lwbs_p
    admit_p = np.array([0, 0.80, 0.55, 0.25, 0.05, 0.01])[triage]
    disposition = np.where(lwbs, "Left without being seen",
                           np.where(rng.random(n) < admit_p, "Admitted", "Discharged"))
    treat = np.array([0, 300, 260, 210, 120, 80])[triage] * rng.gamma(3, 1 / 3, n)
    los = np.where(lwbs, wait, wait + treat.round()).astype(int)

    df = pd.DataFrame({
        "visit_id": np.arange(1, n + 1),
        "date": pd.to_datetime(date),
        "hour": hour,
        "age_group": age,
        "arrival_mode": arrival,
        "triage_level": triage,
        "wait_minutes": wait,
        "length_of_stay_minutes": los,
        "disposition": disposition,
    })
    df.insert(2, "weekday", df["date"].dt.day_name())
    df.insert(3, "month", df["date"].dt.month_name())
    return df

er = make_er_data()
print(f"Riverside Community Hospital (fictional) · {len(er):,} ER visits in 2025")
er.head()

**What's in each column**

| Column | Meaning |
| --- | --- |
| `date`, `weekday`, `month`, `hour` | When the patient arrived |
| `age_group` | 0-17, 18-44, 45-64, 65+ |
| `arrival_mode` | Walk-in or Ambulance |
| `triage_level` | 1 = most urgent … 5 = least urgent |
| `wait_minutes` | Minutes from arrival until seen by a clinician |
| `length_of_stay_minutes` | Minutes from arrival until leaving the ER |
| `disposition` | Admitted, Discharged, or Left without being seen |

## Step 1 · The answer key (computed by code, not by AI)
Code does exactly what it's told, every time. These are the numbers we'll trust.

In [ ]:
days_per_weekday = er.groupby("weekday")["date"].nunique()
visits_per_day = (er.groupby("weekday").size() / days_per_weekday).round(1)
winter = er[er["date"].dt.month.isin([12, 1, 2])]["wait_minutes"].median()
summer = er[er["date"].dt.month.isin([6, 7, 8])]["wait_minutes"].median()

truth = {
    "busiest_weekday": visits_per_day.idxmax(),
    "avg_visits_per_day": round(len(er) / er["date"].nunique(), 1),
    "peak_arrival_hour": int(er.groupby("hour").size().idxmax()),
    "median_wait_minutes": float(er["wait_minutes"].median()),
    "left_without_being_seen_percent": round((er["disposition"] == "Left without being seen").mean() * 100, 1),
    "ambulance_share_percent": round((er["arrival_mode"] == "Ambulance").mean() * 100, 1),
    "winter_vs_summer_wait_increase_percent": round((winter / summer - 1) * 100, 1),
}
pd.Series(truth, name="answer key").to_frame()

## Step 2 · Build the prompt
We don't paste 30,000 raw rows into a chat. We send a **data brief**: a few summary tables plus clear instructions. The prompt uses the RAFTCE pattern you'll practice on Day 2: **R**ole, **A**udience, **F**ormat, **T**one, **C**onstraints, **E**valuation.

In [ ]:
AUDIENCE = "the hospital's operations director, who has 5 minutes and must decide when to add staff"   # ✏️

tables = {
    "Average visits per day, by weekday": visits_per_day.reindex(
        ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]).to_frame("visits_per_day"),
    "Arrivals by hour of day (0 = midnight)": er.groupby("hour").size().to_frame("arrivals"),
    "By month: visits and median wait (minutes)": er.groupby(er["date"].dt.month).agg(
        visits=("visit_id", "count"), median_wait=("wait_minutes", "median")),
    "By triage level (1 = most urgent)": er.groupby("triage_level").agg(
        visits=("visit_id", "count"), median_wait=("wait_minutes", "median"),
        left_without_being_seen_pct=("disposition", lambda s: round((s == "Left without being seen").mean() * 100, 1))),
    "Left without being seen, by wait time": er.assign(
        wait_band=pd.cut(er["wait_minutes"], [-1, 30, 60, 90, 120, 10_000],
                         labels=["0-30", "31-60", "61-90", "91-120", "120+"])).groupby("wait_band", observed=True).agg(
        visits=("visit_id", "count"),
        left_without_being_seen_pct=("disposition", lambda s: round((s == "Left without being seen").mean() * 100, 1))),
    "Arrival mode": er["arrival_mode"].value_counts().to_frame("visits"),
}
brief = "\n\n".join(f"### {name}\n{t.to_csv()}" for name, t in tables.items())

prompt = f"""ROLE: You are a careful healthcare operations analyst.
AUDIENCE: {AUDIENCE}.
TASK: Find the 3 most useful insights in the summary tables below (fictional ER data, Riverside Community Hospital, 2025, {len(er):,} visits).
FORMAT: For each insight give (1) a one-sentence headline that contains the exact number, (2) which table and which numbers support it, (3) the one chart you would put on a slide, (4) your confidence (high/medium/low) and why.
TONE: Plain English, no jargon.
CONSTRAINTS: Use ONLY these tables. Do not invent numbers. If something cannot be answered from them, say "not in the data".
EVALUATION: Before you answer, re-check every number against the tables and list anything you are unsure about.

DATA:
{brief}"""
print(prompt)

## Step 3 · Ask an AI
**Option A (easiest): run the next cell.** It sends the prompt to Gemini through Colab's built-in AI library (free with a Google account; availability can vary by account and region).

**Option B:** copy the printed prompt above and paste it into Claude, ChatGPT or Gemini in another tab.

**Option C (advanced):** download the full CSV with the cell after next, upload it to Claude or ChatGPT, and ask the same question about the raw data.

In [ ]:
# ▶ Option A: ask Gemini from inside Colab
try:
    from google.colab import ai
    answer = ai.generate_text(prompt)
    print(answer)
except Exception as err:
    print("Colab's built-in AI isn't available here:", type(err).__name__)
    print("Use Option B: copy the prompt printed above into Claude, ChatGPT or Gemini.")

In [ ]:
# ▶ Option C: download the full dataset as a CSV (about 2 MB)
er.to_csv("riverside_er_visits_2025_FICTIONAL.csv", index=False)
try:
    from google.colab import files
    files.download("riverside_er_visits_2025_FICTIONAL.csv")
except ImportError:
    print("Saved riverside_er_visits_2025_FICTIONAL.csv")

## Step 4 · Fact-check the AI ✏️
Type the numbers the AI gave you. Leave `None` for anything it didn't say. Then run the cell. A claim passes if it is within 10% of the answer key (or names the right weekday).

In [ ]:
ai_claims = {
    "busiest_weekday": None,                          # ✏️ e.g. "Monday"
    "avg_visits_per_day": None,                       # ✏️ e.g. 95
    "peak_arrival_hour": None,                        # ✏️ e.g. 19  (7 pm)
    "median_wait_minutes": None,                      # ✏️ e.g. 38
    "left_without_being_seen_percent": None,          # ✏️ e.g. 2.5
    "ambulance_share_percent": None,                  # ✏️ e.g. 20
    "winter_vs_summer_wait_increase_percent": None,   # ✏️ e.g. 10
}

def check(name, claimed):
    actual = truth[name]
    if claimed is None:
        return "— (not claimed)"
    if isinstance(actual, str):
        return "✅ correct" if str(claimed).strip().lower() == actual.lower() else "❌ wrong"
    ok = abs(float(claimed) - actual) <= max(0.10 * abs(actual), 1.0)
    return "✅ within 10%" if ok else "❌ off"

report = pd.DataFrame([{"claim": k, "AI said": v, "answer key": truth[k], "verdict": check(k, v)}
                       for k, v in ai_claims.items()])
report

**Discuss with a neighbor**
- Which claims passed? Which failed, and why (a misread table, a rounding slip, a made-up number)?
- Did the AI do what the **Evaluation** line asked (flag uncertainty)?
- Would you put an unverified number on a slide in front of a director? What could it cost?

## Step 5 · Turn one verified insight into a story chart
Only verified numbers go on slides. This chart is built straight from the data, and its headline is computed, so it can't drift from the truth.

In [ ]:
BUSY = 0.95   # ✏️ highlight hours whose median wait is within 5% of the worst hour (try 0.90)

by_hour = er.groupby("hour")["wait_minutes"].median()
hi, lo = by_hour.max(), by_hour.min()
busy = by_hour[by_hour >= BUSY * hi].index
clock = lambda h: f"{h % 12 or 12} {'am' if h % 24 < 12 else 'pm'}"

fig, ax = plt.subplots()
ax.bar(by_hour.index, by_hour.values, color=[CORAL if h in busy else GRAY for h in by_hour.index])
ax.text(by_hour.idxmax(), hi + 1, f"{hi:.0f} min", ha="center", color=CORAL, fontweight="bold")
ax.text(by_hour.idxmin(), lo + 1, f"{lo:.0f} min", ha="center", color=INK)
ax.set_xticks(range(0, 24, 3))
ax.set_xticklabels([clock(h) for h in range(0, 24, 3)])
add_titles(ax, f"From {clock(min(busy))} to {clock(max(busy) + 1)}, median waits hold near {hi:.0f} minutes, vs. {lo:.0f} overnight",
           "Median wait to see a clinician (minutes), by hour of arrival")
add_source(fig, "Source: Riverside Community Hospital ER visits, 2025 (FICTIONAL teaching data)")
plt.show()

## What to remember
- **AI is a fast first analyst, not the final source.** Recompute any number that goes on a slide.
- **Give AI a data brief, not a data dump.** Summaries + clear constraints = fewer invented numbers.
- **Ask it to show its work** ("which table, which numbers?") so you can check it quickly.
- **Disclose** when AI helped make a chart or a claim, and protect private data.